# Week 5 - Apache Spark Fundamentals and Data Processing

**Name:** Imran Alam

**Internship:** Data Engineering Internship

**Week:** 5

## Objective

To understand Apache Spark fundamentals and perform data cleaning, transformation, filtering, aggregation, schema modification, and build a complete data processing pipeline using PySpark DataFrames.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [3]:
spark = SparkSession.builder \
    .appName("Week5_Assignment") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [4]:
df_sales = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

In [5]:
df_sales.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [6]:
df_sales.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



## Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

### Answer

Traditional MapReduce stores the output of every processing stage on disk before starting the next stage. This repeated disk access increases execution time and reduces performance for large-scale data processing. It is also less suitable for applications that require repeated computations, such as machine learning and interactive analytics, because the same data must be read from disk multiple times. Apache Spark overcomes these limitations by performing most computations in memory, which significantly reduces disk I/O and improves processing speed. In addition, Spark provides easy-to-use APIs and supports SQL, streaming, machine learning, and graph processing within a single framework, making it a more efficient solution for modern big data workloads.

## Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

### Answer

Spark improves the performance of iterative machine learning algorithms by storing intermediate data in memory instead of writing it to disk after every operation. During model training, the same dataset is processed repeatedly over multiple iterations. In disk-based systems like MapReduce, each iteration requires reading data from storage again, which increases execution time. Spark keeps frequently used data in memory, allowing subsequent iterations to access it much faster. This reduces processing delays, minimizes disk I/O, and makes Spark highly efficient for machine learning and other iterative data processing tasks.

## Q3. Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

**Note:** The Superstore dataset uses **Customer ID** and **Order Date** instead of **user_id** and **transaction_date**.

In [11]:
# Remove duplicate rows based on Customer ID and Order Date

df_no_duplicates = df_sales.dropDuplicates(["Customer ID", "Order Date"])

df_no_duplicates.show(5)

+------+--------------+----------+---------+--------------+-----------+-------------+--------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|Customer Name| Segment|      Country|         City|     State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+---------+--------------+-----------+-------------+--------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|  1300|CA-2015-121391| 10/4/2015|10/7/2015|   First Class|   AA-10315|   Alex Avila|Consumer|United States|San Francisco|California|      94109|   West|OFF-ST-10001590|Office Supplies|     Storage|Tenex Personal Pr...|   26.96| 

In [12]:
print("Original Records :", df_sales.count())
print("Records After Removing Duplicates :", df_no_duplicates.count())

Original Records : 9994
Records After Removing Duplicates : 4992


## Q4. Given a DataFrame `df_sales`, write a query to filter for rows where the region is **'West'** and then group by **product_category** to find the average **sale_amount**.

**Note:** The Superstore dataset uses the columns `Region`, `Category`, and `Sales` instead of `region`, `product_category`, and `sale_amount`.

In [6]:
from pyspark.sql.functions import col, avg

west_sales = df_sales.filter(col("Region") == "West")

average_sales = west_sales.groupBy("Category").agg(
    avg("Sales").alias("Average_Sales")
)

average_sales.show()

+---------------+----------------+
|       Category|   Average_Sales|
+---------------+----------------+
|Office Supplies|116.422376910912|
|      Furniture|357.302324611033|
|     Technology|420.687532554257|
+---------------+----------------+



## Q5. What is the difference between `.na.drop()` and `.na.fill()`? Provide a code example of filling null values in a status column with the string 'Unknown'.

### Answer

The `.na.drop()` function removes rows that contain null values, whereas `.na.fill()` replaces null values with a specified value without deleting the rows. This helps preserve the dataset while handling missing information.

In [10]:
# Demonstration of .na.drop()

dropped_df = df_sales.na.drop()

print("Original Records :", df_sales.count())
print("Records After na.drop() :", dropped_df.count())

Original Records : 9994
Records After na.drop() : 9994


In [11]:
filled_df = df_sales.na.fill({"Region": "Unknown"})

filled_df.select("Region").show(10)

+------+
|Region|
+------+
| South|
| South|
|  West|
| South|
| South|
|  West|
|  West|
|  West|
|  West|
|  West|
+------+
only showing top 10 rows


**Note:** The provided Superstore dataset does not contain a `status` column. Therefore, the `Region` column is used to demonstrate the `.na.fill()` operation.

## Q6. Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [12]:
from pyspark.sql.functions import count

city_count = df_sales.groupBy("City") \
    .agg(count("*").alias("Total_Records")) \
    .filter(col("Total_Records") > 100)

city_count.show()

+-------------+-------------+
|         City|Total_Records|
+-------------+-------------+
|  Springfield|          163|
|       Dallas|          157|
| Philadelphia|          537|
|  Los Angeles|          747|
|San Francisco|          510|
|    San Diego|          170|
|      Detroit|          115|
|     Columbus|          222|
|      Chicago|          314|
|      Seattle|          428|
|New York City|          915|
|      Houston|          377|
| Jacksonville|          125|
+-------------+-------------+



## Q7. How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

### Answer

Spark DataFrames are immutable, which means the original DataFrame cannot be changed after it is created. Any operation such as dropping a column, renaming a column, or filtering rows creates a new DataFrame instead of modifying the existing one. During data cleaning, the transformed DataFrame must be assigned to a new variable or overwrite the previous variable so that the updated data can be used in later operations.

## Q8. Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

**Note:** The provided Superstore dataset does not contain the columns `age` and `subscription`. Therefore, the following code is provided as a generic PySpark example to demonstrate the required filtering syntax.

In [ ]:
# Generic PySpark example (not executed on the Superstore dataset)

# filtered_df = df.filter(
#     (col("age") >= 18) &
#     (col("age") <= 30) &
#     (col("subscription") == "Premium")
# )

# filtered_df.show()

## Q9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like `sum()` or `avg()`?

### Answer

Handling null values before performing mathematical aggregations helps produce more accurate and reliable results. If null values are ignored or left untreated, the calculated statistics may not correctly represent the dataset. Cleaning missing values beforehand also prevents unexpected results during data processing and improves the overall quality of data analysis.

## Q10. Write the code to revise a column named `raw_timestamp` by casting it to a `TimestampType` and renaming it to `event_time`.

**Note:** The Superstore dataset does not contain the `raw_timestamp` column. The following code demonstrates the required PySpark syntax.

In [ ]:
from pyspark.sql.types import TimestampType

# Generic PySpark example

# df = df.withColumn(
#     "raw_timestamp",
#     col("raw_timestamp").cast(TimestampType())
# )

# df = df.withColumnRenamed(
#     "raw_timestamp",
#     "event_time"
# )

# df.show()

## Q11. Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

### Answer

A shuffle is the process of redistributing data across different partitions so that records with the same key are brought together for operations such as `groupBy()`. During this process, data is transferred between executors over the network, which increases execution time compared to simple transformations. Since data moves across multiple partitions instead of remaining within the same partition, grouping operations are classified as wide transformations.

## Q12. Write a code snippet that identifies and removes rows where the `email` column contains null values OR the `username` is an empty string.

**Note:** The provided Superstore dataset does not contain the `email` and `username` columns. The following code demonstrates the required PySpark syntax.

In [ ]:
# Generic PySpark example

# cleaned_df = df.filter(
#     col("email").isNotNull() &
#     (col("username") != "")
# )

# cleaned_df.show()

## Q13. How do you use the `.agg()` function to calculate multiple statistics at once, such as the minimum, maximum, and mean of the price column?

**Note:** The Superstore dataset uses the `Sales` column instead of `price`.

In [14]:
from pyspark.sql.functions import min, max, avg

df_sales.agg(
    min("Sales").alias("Minimum_Sales"),
    max("Sales").alias("Maximum_Sales"),
    avg("Sales").alias("Average_Sales")
).show()

+-------------+-------------+-----------------+
|Minimum_Sales|Maximum_Sales|    Average_Sales|
+-------------+-------------+-----------------+
|        0.444|     22638.48|229.8580008304938|
+-------------+-------------+-----------------+



## Q14. In the context of cleaning a dataset, what is the risk of using `inferSchema=true` when your source data contains messy or inconsistent date formats?

### Answer

When `inferSchema=true` is used, Spark automatically determines the data type of each column based on the input data. If a date column contains inconsistent formats, Spark may identify it as a string or infer an incorrect data type. This can lead to errors during filtering, sorting, or date calculations. Therefore, it is often better to define the schema manually or clean the data before performing further processing.

## Q15. Write a final processing pipeline that:

- Filters out duplicates.
- Fills null prices with 0.
- Groups by `store_id` to calculate total revenue.

**Note:** The Superstore dataset does not contain the `store_id` and `price` columns. The following code demonstrates the required PySpark syntax.

In [ ]:
# Generic PySpark example

# final_df = (
#     df.dropDuplicates()
#       .na.fill({"price": 0})
#       .groupBy("store_id")
#       .agg(sum("price").alias("Total_Revenue"))
# )

# final_df.show()